# Public Pancreatic Database — Mesenchymal Subtype Annotation
## Quality Control, Concord Embedding, and Pericyte/Fibroblast Sub-clustering

**Input:** `pancdb_clean_final_really.h5ad` → `hadata`
(Pre-processed public pancreatic scRNA-seq database,
multiple donors, pre-labeled with broad cell types
including "mesenchymal cell")

**Output:** `NDpancDB_labeledCV_cleaned_mesleidenclusters.h5ad`
(Same object with refined mesenchymal sub-cluster labels
and pericyte/fibroblast annotations added)

---

### Pipeline Overview

1. Load pre-processed public pancreatic scRNA-seq object
2. QC metric calculation and visualization
3. Log-transform QC metrics → z-score → composite QC score
4. Per-donor QC summary: median score barplot + KDE distributions
5. Library complexity scatter (total counts vs genes detected)
6. Gene detection rate across cells
7. Cells per sample barplot
8. Visualize existing broad cell type labels on Concord UMAP
9. Leiden sub-clustering restricted to mesenchymal cells (res=0.5)
10. Remove low-quality / ambiguous mesenchymal clusters
    (clusters with ≤5 cells, cluster 5, cluster 6)
11. Pairwise DEG analysis between all mesenchymal cluster pairs
    (Wilcoxon, use_raw=False) to assess cluster identities
12. Marker gene UMAP: COL1A1, COL1A2, COL5A1, RGS5, PLVAP
    to confirm pericyte vs fibroblast separation
13. Rename mesenchymal clusters into biological labels:
    cluster 0 → pericytes
    clusters 1–4 → fibroblasts_1 through fibroblasts_4
14. Subset mesenchymal cells only → mes_only
15. Violin plots of collagen markers across clusters
16. Export Prism-ready tables per gene per cluster
17. Save final annotated object

---

### Mesenchymal Sub-cluster → Cell Type Mapping

| mes_leiden | Final Label |
|---|---|
| mesenchymal cell,0 | pericytes |
| mesenchymal cell,1 | fibroblasts_1 |
| mesenchymal cell,2 | fibroblasts_2 |
| mesenchymal cell,3 | fibroblasts_3 |
| mesenchymal cell,4 | fibroblasts_4 |
| mesenchymal cell,5 | Removed |
| mesenchymal cell,6 | Removed |

**Note:** Fibroblast subtypes kept separate intentionally
(not merged) to preserve heterogeneity for downstream
cross-dataset comparison with MERSCOPE vascular labels.

---

### Key Output Columns

| Column | Description |
|---|---|
| `mes_leiden` | Raw Leiden sub-clusters (res=0.5, restricted to mesenchymal cell) |
| `mes_leiden_new` | Refined labels with subtype numbering (fibroblasts_1–4, pericytes) |
| `mes_leiden_new2` | Merged version (all fibroblast subtypes collapsed to "fibroblasts") |

**mes_leiden_new2** is used for simplified comparisons.
**mes_leiden_new** preserves subtype granularity for detailed DEG.

---

### QC Metrics Computed

| Metric | Description |
|---|---|
| `n_genes_by_counts` | Genes detected per cell |
| `total_counts` | Total UMI counts per cell |
| `log_counts` | log10(total_counts + 1) |
| `log_genes` | log10(n_genes + 1) |
| `log_counts_z` | Z-scored log counts |
| `log_genes_z` | Z-scored log genes |
| `qc_score` | Composite: log_counts_z + log_genes_z |

QC score summarized per donor as median; KDE distributions
plotted to identify outlier donors before sub-clustering.

---

### Pairwise DEG Comparisons Run

All comparisons: Wilcoxon, use_raw=False, within mesenchymal
subset only. Top 25 genes plotted per comparison.

| Group A | Group B |
|---|---|
| mesenchymal cell,1 | mesenchymal cell,2 |
| mesenchymal cell,2 | mesenchymal cell,1 |
| mesenchymal cell,2 | mesenchymal cell,3 |
| mesenchymal cell,3 | mesenchymal cell,2 |
| mesenchymal cell,3 | mesenchymal cell,4 |
| mesenchymal cell,4 | mesenchymal cell,3 |
| mesenchymal cell,4 | mesenchymal cell,5 |
| mesenchymal cell,5 | mesenchymal cell,4 |
| mesenchymal cell,5 | mesenchymal cell,0 |
| mesenchymal cell,0 | mesenchymal cell,5 |

Pericyte identity confirmed by cluster 0 enrichment for:
RGS5, PDGFRB, ACE2 relative to all fibroblast clusters.
Fibroblast clusters confirmed by COL1A1, COL1A2, COL5A1,
DCN, LUM enrichment over pericyte cluster.

---

### Marker Genes Used for Annotation

| Marker | Cell Type |
|---|---|
| RGS5, PDGFRB | Pericytes |
| COL1A1, COL1A2, COL5A1, DCN | Fibroblasts |
| PLVAP, VWF | Endothelial (contamination check) |
| GCG | Alpha cells (contamination check) |
| COL4A1, COL4A2 | Basement membrane (shared) |

In [ ]:
import concord as ccd
import scanpy as sc
import anndata as ad
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib as mpl


In [ ]:
hadata = sc.read_h5ad('/Users/kmeneses/Desktop/Fig_plots/pancdb_clean_final_really.h5ad')
hadata

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
sc.pp.calculate_qc_metrics(
    hadata,
    inplace=True
)

In [ ]:
sc.pl.violin(
    hadata,
    ["n_genes_by_counts", "total_counts"],
    groupby="donor_id",
    jitter=0.4,
    multi_panel=True,
    rotation=45
)

In [ ]:
import numpy as np

# log transform (stabilizes scale)
hadata.obs["log_counts"] = np.log10(hadata.obs["total_counts"] + 1)
hadata.obs["log_genes"] = np.log10(hadata.obs["n_genes_by_counts"] + 1)

# library complexity

# z-score normalize each metric
qc_metrics = ["log_counts", "log_genes"]

for m in qc_metrics:
    hadata.obs[f"{m}_z"] = (
        (hadata.obs[m] - hadata.obs[m].mean()) / hadata.obs[m].std()
    )

# combined QC score
hadata.obs["qc_score"] = (
    hadata.obs["log_counts_z"] +
    hadata.obs["log_genes_z"] 
)

In [ ]:
qc_summary = hadata.obs.groupby("donor_id")["qc_score"].median().sort_values()

import matplotlib.pyplot as plt

plt.figure(figsize=(5,2.5))

qc_summary.plot(kind="bar")

plt.ylabel("Median QC score")
plt.xlabel("Sample")
plt.title("Median cell quality per donor")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5,3))

for donor in hadata.obs["donor_id"].unique():
    subset = hadata.obs[hadata.obs["donor_id"] == donor]
    sns.kdeplot(subset["qc_score"], label=donor, alpha=0.3)

plt.xlabel("QC score")
plt.title("QC distribution per donor")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(4,3))

sns.scatterplot(
    data=hadata.obs,
    x="total_counts",
    y="n_genes_by_counts",
    s=3,
    alpha=0.3
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Total counts")
plt.ylabel("Number of genes")
plt.title("Library complexity relationship")

plt.tight_layout()
plt.show()

In [ ]:
gene_detection = (hadata.X > 0).mean(axis=0)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# COUNT CELLS PER SAMPLE
# =========================
sample_counts = (
    hadata.obs["donor_id"]
    .value_counts()
    .sort_index()   # keeps donor order clean
)

# Convert to dataframe (optional, useful for export)
df_counts = sample_counts.reset_index()
df_counts.columns = ["donor_id", "n_cells"]

# =========================
# PLOT
# =========================
plt.figure(figsize=(4,2.5))

plt.bar(
    df_counts["donor_id"],
    df_counts["n_cells"]
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cells")
plt.xlabel("Sample")
plt.title("Cells per sample")

plt.tight_layout()

# =========================
# SAVE
# =========================


plt.show()

In [ ]:
hadata

In [ ]:

# Plot the UMAP embeddings
color_by = ['cell_type_grouped', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    hadata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
sc.pp.calculate_qc_metrics(hadata_clean, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    hadata_clean,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
hadata_clean

In [ ]:
sc.pp.neighbors(hadata, use_rep='Concord')
sc.tl.leiden(
    hadata,
    resolution=0.5,
    restrict_to=("cell_type", ["mesenchymal cell"]),
    key_added="mes_leiden"
)

In [ ]:
color_by = ['cell_type', 'mes_leiden'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    hadata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
hadata.obs['mes_leiden'].value_counts()

In [ ]:
# remove Leiden groups with <=1 cell
counts = hadata.obs["mes_leiden"].value_counts()

keep_clusters = counts[counts > 5].index

hadata = hadata[hadata.obs["mes_leiden"].isin(keep_clusters)].copy()

In [ ]:
sc.tl.rank_genes_groups(hadata, "mes_leiden", use_raw=False)
sc.pl.rank_genes_groups(hadata, gene_symbols='features')

In [ ]:
sc.pl.embedding(hadata,
                basis="Concord_UMAP",
                 color=["mes_leiden"])

In [ ]:
# subset only mesenchymal Leiden clusters
mes = hadata[
    hadata.obs["mes_leiden"].astype(str).str.contains("mesenchymal cell")
].copy()

# run differential expression between mesenchymal clusters
sc.tl.rank_genes_groups(
    mes,
    groupby="mes_leiden",
    method="wilcoxon",
    use_raw=False
)

# optional: visualize top markers
sc.pl.rank_genes_groups(
    mes,
    n_genes=20,
    sharey=False
)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,1",
        "mesenchymal cell,2"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,1"],
    reference="mesenchymal cell,2",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,1"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,1",
        "mesenchymal cell,2"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,2"],
    reference="mesenchymal cell,1",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,2"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,3",
        "mesenchymal cell,2"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,2"],
    reference="mesenchymal cell,3",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,2"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,3",
        "mesenchymal cell,2"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,3"],
    reference="mesenchymal cell,2",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,3"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,3",
        "mesenchymal cell,4"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,3"],
    reference="mesenchymal cell,4",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,3"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,3",
        "mesenchymal cell,4"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,4"],
    reference="mesenchymal cell,3",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,4"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,4",
        "mesenchymal cell,5"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,4"],
    reference="mesenchymal cell,5",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,4"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,4",
        "mesenchymal cell,5"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,5"],
    reference="mesenchymal cell,4",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,5"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,0",
        "mesenchymal cell,5"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,5"],
    reference="mesenchymal cell,0",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,5"
)

deg.head(20)

In [ ]:
# run DE specifically between mesenchymal cell,4 and mesenchymal cell,5

subset = mes[
    mes.obs["mes_leiden"].isin([
        "mesenchymal cell,5",
        "mesenchymal cell,0"
    ])
].copy()

sc.tl.rank_genes_groups(
    subset,
    groupby="mes_leiden",
    groups=["mesenchymal cell,0"],
    reference="mesenchymal cell,5",
    method="wilcoxon",
    use_raw=False
)

# visualize top DE genes
sc.pl.rank_genes_groups(
    subset,
    n_genes=25,
    sharey=False
)

# export DE table
deg = sc.get.rank_genes_groups_df(
    subset,
    group="mesenchymal cell,0"
)

deg.head(20)

In [ ]:
sc.pl.embedding(
    mes,
    basis="Concord_UMAP",
    color=["mes_leiden", "COL1A1", "COL1A2", "COL5A1", "GCG", "PLVAP", "RGS5"],
    gene_symbols="features",
    size=8,
    ncols=2,
    wspace=0.3,
    use_raw=False
)

In [ ]:
sc.pl.embedding(hadata,
                basis="Concord_UMAP",
                 color=["mes_leiden"])

In [ ]:
# remove mesenchymal cell,6
hadata = hadata[
    hadata.obs["mes_leiden"] != "mesenchymal cell,6"
].copy()

# remove unused categories
hadata.obs["mes_leiden"] = (
    hadata.obs["mes_leiden"]
    .cat.remove_unused_categories()
)

In [ ]:
# remove mesenchymal cell,6
hadata = hadata[
    hadata.obs["mes_leiden"] != "mesenchymal cell,5"
].copy()

# remove unused categories
hadata.obs["mes_leiden"] = (
    hadata.obs["mes_leiden"]
    .cat.remove_unused_categories()
)

In [ ]:
# rename mesenchymal clusters but keep all other cell types unchanged

rename_dict = {
    "mesenchymal cell,0": "pericytes",
    "mesenchymal cell,1": "fibroblasts",
    "mesenchymal cell,2": "fibroblasts",
    "mesenchymal cell,3": "fibroblasts",
    "mesenchymal cell,4": "fibroblasts",}

# start from original cell labels
hadata.obs["mes_leiden_new2"] = hadata.obs["cell_type"].astype(str)

# replace only mesenchymal cluster cells
mask = hadata.obs["mes_leiden"].astype(str).isin(rename_dict.keys())

hadata.obs.loc[mask, "mes_leiden_new2"] = (
    hadata.obs.loc[mask, "mes_leiden"]
    .astype(str)
    .map(rename_dict)
    .values
)

# optional
hadata.obs["mes_leiden_new2"] = (
    hadata.obs["mes_leiden_new2"]
    .astype("category")
)

In [ ]:
sc.pl.embedding(hadata,
                basis="Concord_UMAP",
                 color=["mes_leiden_new2"])

In [ ]:
# rename mesenchymal clusters but keep all other cell types unchanged

rename_dict = {
    "mesenchymal cell,0": "pericytes",
    "mesenchymal cell,1": "fibroblasts_1",
    "mesenchymal cell,2": "fibroblasts_2",
    "mesenchymal cell,3": "fibroblasts_3",
    "mesenchymal cell,4": "fibroblasts_4"}

# start from original cell labels
hadata.obs["mes_leiden_new"] = hadata.obs["cell_type"].astype(str)

# replace only mesenchymal cluster cells
mask = hadata.obs["mes_leiden"].astype(str).isin(rename_dict.keys())

hadata.obs.loc[mask, "mes_leiden_new"] = (
    hadata.obs.loc[mask, "mes_leiden"]
    .astype(str)
    .map(rename_dict)
    .values
)

# optional
hadata.obs["mes_leiden_new"] = (
    hadata.obs["mes_leiden_new"]
    .astype("category")
)

In [ ]:
sc.pl.embedding(
    hadata,
    basis="Concord_UMAP",
    color=["mes_leiden_new", "COL1A1", "COL1A2", "COL5A1", "GCG", "PLVAP", "RGS5"],
    gene_symbols="features",
    size=8,
    ncols=2,
    wspace=0.3,
    use_raw=False
)

In [ ]:
# =========================================================
# KEEP ONLY:
# - pericytes
# - fibroblasts_1
# - fibroblasts_2
# - fibroblasts_3
# - fibroblasts_4
#
# KEEP SUBTYPES SEPARATE
# NO MERGING
# =========================================================

keep_mes = [

    "pericytes",

    "fibroblasts_1",
    "fibroblasts_2",
    "fibroblasts_3",
    "fibroblasts_4"
]

# =========================================================
# SUBSET
# =========================================================

mes_only = hadata[
    hadata.obs["mes_leiden_new"].isin(keep_mes)
].copy()

# =========================================================
# CHECK
# =========================================================

print(
    mes_only.obs["mes_leiden_new"]
    .value_counts()
)

print(mes_only)

In [ ]:
sc.pl.embedding(
    mes_only,
    basis="Concord_UMAP",
    color=["mes_leiden_new", "COL1A1", "COL1A2", "COL5A1", "GCG", "PLVAP", "RGS5", "DCN"],
    gene_symbols="features",
    size=20,
    ncols=2,
    wspace=0.3,
    use_raw=False
)

In [ ]:
sc.pl.embedding(hadata,
                basis="Concord_UMAP",
                 color=["mes_leiden_new"])

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

genes = ["COL4A1", "COL4A2", "COL1A1", "COL5A1"]

import pandas as pd


# get expression matrix
expr = mes[:, genes].to_df()

# add cluster labels
expr["mes_leiden"] = mes.obs["mes_leiden"].values
prism_tables = {}

for gene in genes:
    
    df = expr[[gene, "mes_leiden"]].copy()
    
    prism = pd.DataFrame()
    
    for cluster in sorted(df["mes_leiden"].unique()):
        prism[f"{gene}_cluster_{cluster}"] = (
            df.loc[df["mes_leiden"] == cluster, gene]
            .reset_index(drop=True)
        )
    
    prism_tables[gene] = prism
    


print("Prism tables exported.")

sc.pl.violin(
    mes,
    genes,
    groupby="mes_leiden",
    stripplot=False,
    jitter=False,
    use_raw=False
)

In [ ]:
hadata.write_h5ad('/Users/kmeneses/Desktop/updated_data/NDpancDB_labeledCV_cleaned_mesleidenclusters.h5ad')

In [ ]:
hadata